# Scaled Dot-Product Self-Attention in NumPy

This notebook builds self-attention **from scratch**: no deep learning
framework, just NumPy for every numeric operation (pandas is used only
to *display* results as labeled tables).

**Pipeline:** sentence &rarr; tokens &rarr; embeddings `X` &rarr; `Q, K, V`
projections &rarr; scaled dot-product scores &rarr; row-wise softmax &rarr;
weighted sum of `V`.

The notebook is organized as:
1. Setup (imports, seed, embeddings, projection matrices)
2. Implementation (`tokenize`, `embed`, `softmax`, `scaled_dot_product_attention`, `run_attention`)
3. Demo run
4. Test cases (with automatic checks)
5. Report (written explanation of the mechanism)


## Section 1 — Setup

In [1]:
import numpy as np
import re
import pandas as pd  # only used to *display* results as labeled tables

# A single fixed seed drives every source of randomness in this notebook,
# so re-running top to bottom always reproduces the exact same numbers.
SEED = 42

D_MODEL = 8   # dimension of each token's embedding vector
D_K = 8       # dimension Q and K are projected into (must match for the dot product)
D_V = 8       # dimension V is projected into (this is also the output dimension)

**Embedding source.** No pretrained embedding table is provided, so we
build one ourselves: every unique word gets a random vector, but the
vector is derived from a seed computed *from the word's own characters*
(combined with the global `SEED`) rather than from the order in which the
word happens to appear. That means "trophy" gets the same vector whether
it's the 1st word we ever see or the 50th, and whether it shows up in the
demo sentence or in a brand-new test sentence typed in later — which is
exactly the reproducibility the task asks for.

In [2]:
_embedding_cache = {}

def _word_seed(word: str, seed: int = SEED) -> int:
    '''Turn a word into a deterministic 32-bit seed. Each character's
    ordinal value (weighted by its position) is folded into a running
    hash, so the seed depends only on the word's spelling, not on when
    it was first looked up.'''
    h = seed
    for i, ch in enumerate(word):
        h = (h * 31 + ord(ch) * (i + 1)) % (2 ** 32)
    return h

def get_embedding(word: str, dim: int = D_MODEL) -> np.ndarray:
    '''Return a fixed, reproducible embedding vector for `word`.
    Results are cached so repeated lookups are cheap and return the
    identical array.'''
    key = word.lower()
    if key not in _embedding_cache:
        rng = np.random.RandomState(_word_seed(key))
        _embedding_cache[key] = rng.randn(dim)
    return _embedding_cache[key]

**Projection matrices `W_Q`, `W_K`, `W_V`.** Each is initialized once,
from its own fixed seed, and reused for every sentence in the notebook.
We scale them down (`* 0.1`) purely so the raw dot products don't start
out huge before scaling — the projections themselves are still random,
untrained weights, exactly as the task describes.

In [3]:
_rng_q = np.random.RandomState(SEED + 1)
_rng_k = np.random.RandomState(SEED + 2)
_rng_v = np.random.RandomState(SEED + 3)

W_Q = _rng_q.randn(D_MODEL, D_K) * 0.1
W_K = _rng_k.randn(D_MODEL, D_K) * 0.1
W_V = _rng_v.randn(D_MODEL, D_V) * 0.1

print("W_Q shape:", W_Q.shape)
print("W_K shape:", W_K.shape)
print("W_V shape:", W_V.shape)

W_Q shape: (8, 8)
W_K shape: (8, 8)
W_V shape: (8, 8)


## Section 2 — Implementation

In [4]:
def tokenize(sentence: str):
    '''Lowercase the sentence, then split it into word tokens and
    punctuation tokens. Internal apostrophes ("didn't", "it's") are kept
    as part of the word; every other punctuation mark becomes its own
    token, which is what lets "big." split into "big" and ".".'''
    sentence = sentence.lower()
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?|[^\sa-z]", sentence)
    return tokens

tokenize("The trophy didn't fit in the suitcase because it was too big.")

['the',
 'trophy',
 "didn't",
 'fit',
 'in',
 'the',
 'suitcase',
 'because',
 'it',
 'was',
 'too',
 'big',
 '.']

In [5]:
def embed(tokens):
    '''Look up each token's embedding vector and stack them row-wise
    into a single (n, d_model) matrix — this is X.'''
    return np.vstack([get_embedding(tok) for tok in tokens])

In [6]:
def softmax(x, axis=-1):
    '''Numerically stable softmax. Subtracting the row max before
    exponentiating doesn't change the result (softmax is shift-invariant)
    but keeps exp() from overflowing on large scores. Dividing by the row
    sum afterward is what makes every row sum to exactly 1.'''
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x_shifted)
    return e / np.sum(e, axis=axis, keepdims=True)

In [7]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    '''
    Q: (n, d_k)   K: (n, d_k)   V: (n, d_v)
    Returns (output, weights):
      output  -> (n, d_v)  a new representation for every token
      weights -> (n, n)    how much each token attends to every other token
    '''
    d_k = Q.shape[-1]

    # Raw similarity between every query and every key, then scaled down
    # by sqrt(d_k) so the dot products don't grow (and blow up softmax)
    # just because the vectors are longer.
    scores = Q @ K.T / np.sqrt(d_k)

    if mask is not None:
        # Positions we want to hide (mask == 0) are pushed to a huge
        # negative number so softmax turns them into ~0 probability.
        scores = np.where(mask == 0, -1e9, scores)

    # Row-wise softmax: for each token, turn its scores against every
    # other token into a probability distribution that sums to 1.
    weights = softmax(scores, axis=-1)

    # Blend the Value vectors according to those attention probabilities.
    output = weights @ V
    return output, weights

In [8]:
def run_attention(sentence: str, mask=None):
    '''Single entry point: takes a raw sentence string and runs the
    entire pipeline, returning every intermediate array plus the final
    output — this is what the demo and test cells call.'''
    tokens = tokenize(sentence)                 # 1. tokenize
    X = embed(tokens)                            # 2. embed -> (n, d_model)

    Q = X @ W_Q                                  # 3. project into queries
    K = X @ W_K                                  #    project into keys
    V = X @ W_V                                  #    project into values

    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)              # 4. scaled scores (n, n)
    output, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
    # 5-6. softmax + weighted sum happen inside scaled_dot_product_attention

    return {
        "tokens": tokens,
        "X": X, "Q": Q, "K": K, "V": V,
        "scores": scores, "weights": weights, "output": output,
    }

## Section 3 — Demo Run

In [9]:
demo_sentence = "The trophy didn't fit in the suitcase because it was too big."
result = run_attention(demo_sentence)

print("Sentence:", demo_sentence)
print("Tokens:  ", result["tokens"])
print()
print("X shape:     ", result["X"].shape)
print("Q shape:     ", result["Q"].shape)
print("K shape:     ", result["K"].shape)
print("V shape:     ", result["V"].shape)
print("scores shape:", result["scores"].shape)
print("weights shape:", result["weights"].shape)
print("output shape: ", result["output"].shape)

Sentence: The trophy didn't fit in the suitcase because it was too big.
Tokens:   ['the', 'trophy', "didn't", 'fit', 'in', 'the', 'suitcase', 'because', 'it', 'was', 'too', 'big', '.']

X shape:      (13, 8)
Q shape:      (13, 8)
K shape:      (13, 8)
V shape:      (13, 8)
scores shape: (13, 13)
weights shape: (13, 13)
output shape:  (13, 8)


In [10]:
def weights_table(tokens, weights):
    '''Attention weights as a DataFrame labeled with the tokens on
    both axes: row = the attending token, column = the token attended to.'''
    return pd.DataFrame(weights, index=tokens, columns=tokens).round(3)

weights_table(result["tokens"], result["weights"])

,the,trophy,didn't,fit,in,the,suitcase,because,it,was,too,big,.
the,0.078,0.078,0.074,0.076,0.080,0.078,0.072,0.075,0.076,0.081,0.078,0.075,0.080
trophy,0.075,0.079,0.077,0.074,0.076,0.075,0.082,0.078,0.079,0.075,0.079,0.076,0.076
didn't,0.084,0.073,0.079,0.076,0.072,0.084,0.076,0.070,0.082,0.078,0.073,0.077,0.076
fit,0.092,0.063,0.089,0.070,0.061,0.092,0.090,0.068,0.093,0.068,0.075,0.073,0.068
in,0.074,0.079,0.077,0.079,0.078,0.074,0.077,0.079,0.076,0.076,0.074,0.079,0.077
the,0.078,0.078,0.074,0.076,0.080,0.078,0.072,0.075,0.076,0.081,0.078,0.075,0.080
suitcase,0.071,0.084,0.072,0.080,0.086,0.071,0.071,0.081,0.068,0.080,0.076,0.078,0.081
because,0.078,0.076,0.082,0.078,0.074,0.078,0.081,0.078,0.077,0.073,0.072,0.079,0.075
it,0.068,0.083,0.068,0.089,0.090,0.068,0.062,0.083,0.063,0.086,0.077,0.081,0.083
was,0.083,0.075,0.083,0.070,0.070,0.083,0.084,0.071,0.087,0.072,0.072,0.076,0.075


In [11]:
def most_attended(tokens, weights):
    '''For each token (row), find the column with the highest weight —
    i.e. the token it attends to most.'''
    idx = np.argmax(weights, axis=-1)
    return {tokens[i]: tokens[j] for i, j in enumerate(idx)}

for tok, target in most_attended(result["tokens"], result["weights"]).items():
    print(f"{tok!r:>12} -> {target!r}")

       'the' -> 'was'
    'trophy' -> 'suitcase'
    "didn't" -> 'the'
       'fit' -> 'it'
        'in' -> 'fit'
  'suitcase' -> 'in'
   'because' -> "didn't"
        'it' -> 'in'
       'was' -> 'it'
       'too' -> 'trophy'
       'big' -> 'it'
         '.' -> 'it'


In [12]:
print("Output vectors (each row is one token's new, context-mixed representation):")
pd.DataFrame(result["output"], index=result["tokens"]).round(3)

Output vectors (each row is one token's new, context-mixed representation):


,0,1,2,3,4,5,6,7
the,-0.010,0.041,-0.026,-0.031,0.037,0.063,-0.051,-0.006
trophy,-0.003,0.031,-0.024,-0.031,0.042,0.063,-0.044,-0.002
didn't,-0.008,0.040,-0.022,-0.022,0.043,0.079,-0.050,-0.005
fit,0.002,0.033,-0.012,-0.020,0.054,0.112,-0.045,-0.002
in,-0.006,0.036,-0.027,-0.024,0.040,0.055,-0.045,-0.005
the,-0.010,0.041,-0.026,-0.031,0.037,0.063,-0.051,-0.006
suitcase,-0.011,0.039,-0.030,-0.030,0.035,0.043,-0.047,-0.005
because,-0.003,0.034,-0.024,-0.020,0.044,0.068,-0.043,-0.003
it,-0.019,0.050,-0.038,-0.027,0.032,0.022,-0.051,-0.012
was,0.000,0.028,-0.017,-0.024,0.044,0.089,-0.045,0.001


## Section 4 — Test Cases

`report(sentence)` reruns the whole pipeline on a new sentence, prints
the same outputs as the demo above, and finishes with automatic checks:

- every row of the attention weights is non-negative and sums to 1
- the output shape is `(n, d_v)`

Because `W_Q`, `W_K`, `W_V` and every word's embedding are fixed once in
Section 1, you can call `report("any new sentence")` in a fresh cell at
any time — live, without changing any code above — and it will just work.

In [13]:
def report(sentence, mask=None):
    r = run_attention(sentence, mask=mask)
    tokens, weights, output = r["tokens"], r["weights"], r["output"]

    print("Sentence:", sentence)
    print("Tokens:  ", tokens)
    print("X shape:", r["X"].shape, "| Q shape:", r["Q"].shape,
          "| K shape:", r["K"].shape, "| V shape:", r["V"].shape)
    print("scores shape:", r["scores"].shape, "| weights shape:", weights.shape,
          "| output shape:", output.shape)

    print("\nAttention weights:")
    display(weights_table(tokens, weights))

    print("\nMost-attended token per token:")
    for tok, target in most_attended(tokens, weights).items():
        print(f"  {tok!r:>12} -> {target!r}")

    print("\nOutput vectors:")
    display(pd.DataFrame(output, index=tokens).round(3))

    # --- automatic checks ---
    assert np.all(weights >= 0), "Found a negative attention weight"
    row_sums = weights.sum(axis=-1)
    assert np.allclose(row_sums, 1.0), f"Rows don't sum to 1: {row_sums}"
    assert output.shape == (len(tokens), D_V), f"Unexpected output shape {output.shape}"
    print("\nAll checks passed: weights are non-negative, rows sum to 1, output shape is correct.")

    return r

In [14]:
_ = report("The trophy didn't fit in the suitcase because it was too big.")

Sentence: The trophy didn't fit in the suitcase because it was too big.
Tokens:   ['the', 'trophy', "didn't", 'fit', 'in', 'the', 'suitcase', 'because', 'it', 'was', 'too', 'big', '.']
X shape: (13, 8) | Q shape: (13, 8) | K shape: (13, 8) | V shape: (13, 8)
scores shape: (13, 13) | weights shape: (13, 13) | output shape: (13, 8)

Attention weights:


,the,trophy,didn't,fit,in,the,suitcase,because,it,was,too,big,.
the,0.078,0.078,0.074,0.076,0.080,0.078,0.072,0.075,0.076,0.081,0.078,0.075,0.080
trophy,0.075,0.079,0.077,0.074,0.076,0.075,0.082,0.078,0.079,0.075,0.079,0.076,0.076
didn't,0.084,0.073,0.079,0.076,0.072,0.084,0.076,0.070,0.082,0.078,0.073,0.077,0.076
fit,0.092,0.063,0.089,0.070,0.061,0.092,0.090,0.068,0.093,0.068,0.075,0.073,0.068
in,0.074,0.079,0.077,0.079,0.078,0.074,0.077,0.079,0.076,0.076,0.074,0.079,0.077
the,0.078,0.078,0.074,0.076,0.080,0.078,0.072,0.075,0.076,0.081,0.078,0.075,0.080
suitcase,0.071,0.084,0.072,0.080,0.086,0.071,0.071,0.081,0.068,0.080,0.076,0.078,0.081
because,0.078,0.076,0.082,0.078,0.074,0.078,0.081,0.078,0.077,0.073,0.072,0.079,0.075
it,0.068,0.083,0.068,0.089,0.090,0.068,0.062,0.083,0.063,0.086,0.077,0.081,0.083
was,0.083,0.075,0.083,0.070,0.070,0.083,0.084,0.071,0.087,0.072,0.072,0.076,0.075



Most-attended token per token:
         'the' -> 'was'
      'trophy' -> 'suitcase'
      "didn't" -> 'the'
         'fit' -> 'it'
          'in' -> 'fit'
    'suitcase' -> 'in'
     'because' -> "didn't"
          'it' -> 'in'
         'was' -> 'it'
         'too' -> 'trophy'
         'big' -> 'it'
           '.' -> 'it'

Output vectors:


,0,1,2,3,4,5,6,7
the,-0.010,0.041,-0.026,-0.031,0.037,0.063,-0.051,-0.006
trophy,-0.003,0.031,-0.024,-0.031,0.042,0.063,-0.044,-0.002
didn't,-0.008,0.040,-0.022,-0.022,0.043,0.079,-0.050,-0.005
fit,0.002,0.033,-0.012,-0.020,0.054,0.112,-0.045,-0.002
in,-0.006,0.036,-0.027,-0.024,0.040,0.055,-0.045,-0.005
the,-0.010,0.041,-0.026,-0.031,0.037,0.063,-0.051,-0.006
suitcase,-0.011,0.039,-0.030,-0.030,0.035,0.043,-0.047,-0.005
because,-0.003,0.034,-0.024,-0.020,0.044,0.068,-0.043,-0.003
it,-0.019,0.050,-0.038,-0.027,0.032,0.022,-0.051,-0.012
was,0.000,0.028,-0.017,-0.024,0.044,0.089,-0.045,0.001



All checks passed: weights are non-negative, rows sum to 1, output shape is correct.


In [15]:
_ = report("The cat sat on the mat.")

Sentence: The cat sat on the mat.
Tokens:   ['the', 'cat', 'sat', 'on', 'the', 'mat', '.']
X shape: (7, 8) | Q shape: (7, 8) | K shape: (7, 8) | V shape: (7, 8)
scores shape: (7, 7) | weights shape: (7, 7) | output shape: (7, 8)

Attention weights:


,the,cat,sat,on,the,mat,.
the,0.141,0.154,0.133,0.139,0.141,0.149,0.143
cat,0.148,0.129,0.156,0.139,0.148,0.138,0.141
sat,0.129,0.155,0.134,0.161,0.129,0.143,0.148
on,0.133,0.131,0.160,0.158,0.133,0.137,0.149
the,0.141,0.154,0.133,0.139,0.141,0.149,0.143
mat,0.146,0.151,0.136,0.137,0.146,0.146,0.138
.,0.146,0.141,0.147,0.136,0.146,0.142,0.141



Most-attended token per token:
         'the' -> 'cat'
         'cat' -> 'sat'
         'sat' -> 'on'
          'on' -> 'sat'
         'mat' -> 'cat'
           '.' -> 'sat'

Output vectors:


,0,1,2,3,4,5,6,7
the,-0.061,0.047,0.038,-0.091,-0.047,0.203,-0.035,0.007
cat,-0.045,0.025,0.046,-0.086,-0.033,0.217,-0.023,0.019
sat,-0.063,0.042,0.036,-0.101,-0.041,0.169,-0.029,0.011
on,-0.046,0.018,0.043,-0.095,-0.030,0.181,-0.017,0.023
the,-0.061,0.047,0.038,-0.091,-0.047,0.203,-0.035,0.007
mat,-0.060,0.046,0.040,-0.089,-0.043,0.211,-0.033,0.007
.,-0.052,0.035,0.043,-0.087,-0.038,0.213,-0.029,0.013



All checks passed: weights are non-negative, rows sum to 1, output shape is correct.


In [16]:
_ = report("She quickly finished her homework before the deadline.")

Sentence: She quickly finished her homework before the deadline.
Tokens:   ['she', 'quickly', 'finished', 'her', 'homework', 'before', 'the', 'deadline', '.']
X shape: (9, 8) | Q shape: (9, 8) | K shape: (9, 8) | V shape: (9, 8)
scores shape: (9, 9) | weights shape: (9, 9) | output shape: (9, 8)

Attention weights:


,she,quickly,finished,her,homework,before,the,deadline,.
she,0.098,0.111,0.125,0.110,0.113,0.110,0.097,0.116,0.119
quickly,0.098,0.130,0.110,0.122,0.111,0.115,0.101,0.099,0.115
finished,0.116,0.107,0.111,0.111,0.111,0.108,0.117,0.109,0.110
her,0.108,0.106,0.117,0.112,0.109,0.112,0.114,0.111,0.113
homework,0.096,0.130,0.109,0.114,0.117,0.121,0.087,0.111,0.112
before,0.105,0.121,0.107,0.118,0.110,0.119,0.099,0.111,0.110
the,0.111,0.108,0.116,0.105,0.115,0.105,0.112,0.115,0.114
deadline,0.112,0.113,0.112,0.120,0.107,0.105,0.113,0.105,0.114
.,0.110,0.109,0.117,0.116,0.108,0.104,0.120,0.102,0.116



Most-attended token per token:
         'she' -> 'finished'
     'quickly' -> 'quickly'
    'finished' -> 'the'
         'her' -> 'finished'
    'homework' -> 'quickly'
      'before' -> 'quickly'
         'the' -> 'finished'
    'deadline' -> 'her'
           '.' -> 'the'

Output vectors:


,0,1,2,3,4,5,6,7
she,0.049,-0.124,0.010,-0.262,-0.020,0.110,-0.030,0.110
quickly,0.060,-0.143,0.010,-0.260,-0.014,0.089,-0.022,0.119
finished,0.056,-0.126,0.015,-0.259,-0.017,0.132,-0.025,0.106
her,0.054,-0.124,0.014,-0.256,-0.018,0.126,-0.026,0.106
homework,0.055,-0.142,0.008,-0.274,-0.008,0.079,-0.021,0.122
before,0.057,-0.138,0.010,-0.266,-0.009,0.097,-0.020,0.115
the,0.050,-0.121,0.014,-0.264,-0.018,0.130,-0.030,0.107
deadline,0.058,-0.131,0.013,-0.254,-0.018,0.121,-0.025,0.109
.,0.057,-0.127,0.014,-0.250,-0.023,0.130,-0.028,0.106



All checks passed: weights are non-negative, rows sum to 1, output shape is correct.


**Bonus: using the `mask` argument.** A mask lets you hide certain
positions from attention (e.g. a causal mask that stops a token from
looking at future tokens). Passing a lower-triangular mask below shows
each token attending only to itself and the tokens before it.

In [17]:
sentence = "Attention changes with context."
tokens = tokenize(sentence)
n = len(tokens)
causal_mask = np.tril(np.ones((n, n)))  # 1s on/below the diagonal, 0s above

_ = report(sentence, mask=causal_mask)

Sentence: Attention changes with context.
Tokens:   ['attention', 'changes', 'with', 'context', '.']
X shape: (5, 8) | Q shape: (5, 8) | K shape: (5, 8) | V shape: (5, 8)
scores shape: (5, 5) | weights shape: (5, 5) | output shape: (5, 8)

Attention weights:


,attention,changes,with,context,.
attention,1.000,0.000,0.000,0.00,0.000
changes,0.493,0.507,0.000,0.00,0.000
with,0.330,0.344,0.326,0.00,0.000
context,0.251,0.270,0.248,0.23,0.000
.,0.203,0.184,0.222,0.19,0.201



Most-attended token per token:
   'attention' -> 'attention'
     'changes' -> 'changes'
        'with' -> 'changes'
     'context' -> 'changes'
           '.' -> 'with'

Output vectors:


,0,1,2,3,4,5,6,7
attention,-0.083,0.045,0.012,-0.236,-0.061,0.016,-0.168,0.067
changes,-0.008,0.079,-0.005,-0.106,0.046,-0.035,-0.014,-0.010
with,0.034,-0.016,0.000,-0.079,-0.123,-0.040,-0.013,-0.015
context,-0.060,0.103,-0.062,-0.114,-0.112,-0.205,-0.020,-0.100
.,-0.059,0.077,-0.064,-0.117,-0.159,-0.137,-0.093,-0.055



All checks passed: weights are non-negative, rows sum to 1, output shape is correct.


## Section 5 — Report

**What problem does attention solve?**
A fixed-size embedding gives a word one vector no matter what sentence
it's in, so it can't capture that meaning depends on context. In *"The
trophy didn't fit in the suitcase because it was too big"*, the word
"it" is ambiguous on its own — attention lets "it" look at every other
word in the sentence and pull in information from whichever ones are
actually relevant (here, mostly "trophy" and "suitcase"), so the model
ends up with a representation of "it" that is shaped by its context
instead of being a single static vector.

**What do Q, K and V represent, and why three separate projections?**
Every token's embedding is projected three different ways because it
needs to play three different roles in the same calculation:
- **Query (Q)** — what this token is "asking for": the kind of
  information it wants to gather from the rest of the sentence.
- **Key (K)** — what this token "advertises" about itself: the label
  other tokens' queries get compared against to decide relevance.
- **Value (V)** — the actual content this token contributes once it has
  been selected as relevant.
If we reused the same vector for all three roles, a token's own
"relevance signature" (key) and the "content it hands over" (value)
would be forced to be identical, which is unnecessarily restrictive.
Separate learned (here, randomly initialized) projections let the model
decide independently how a token should be found versus what it should
contribute once found.

**How does the calculation work, step by step?**
1. **Dot product (`Q @ K.T`)** — for every pair of tokens, the dot
   product between one token's query and another's key measures how
   well they match; a large value means "this key is relevant to what
   that query is looking for."
2. **Scaling (`/ sqrt(d_k)`)** — as the vector dimension `d_k` grows, raw
   dot products tend to grow with it, which pushes softmax into regions
   with extremely small gradients. Dividing by `sqrt(d_k)` keeps the
   scores in a stable range regardless of dimensionality.
3. **Softmax (row-wise)** — turns each token's raw scores against every
   other token into a proper probability distribution (non-negative,
   sums to 1), so the scores become interpretable as "how much attention
   to pay" rather than arbitrary numbers.
4. **Weighted sum (`weights @ V`)** — the output for each token is a
   blend of every token's Value vector, weighted by how much attention
   was paid to it. Tokens that scored high get more of their content
   mixed into the result; tokens that scored near zero contribute almost
   nothing.

**The key distinction.** Q and K together decide *where to look* — they
produce the attention weights, i.e. which tokens matter to which. V
decides *what information is taken* once that decision has been made —
it's the actual content that gets copied (in weighted proportion) into
the output. Changing V without changing Q/K would change *what* gets
passed along without changing *who's paying attention to whom*; changing
Q/K without changing V would redirect attention without changing what
content is available to draw from.
